In [6]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="google.protobuf")

import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import joblib
from collections import deque, Counter

# ==============================================================================
# 1. CARREGAMENTO DO MODELO E CONFIGURAÇÃO DE SUAVIZAÇÃO
# ==============================================================================
caminho_modelo = "modelo_asl_rf.pkl"
#caminho_modelo = "modelo_asl_rf.pkl"
print(f"Carregando modelo de: {caminho_modelo}...")
dados_modelo = joblib.load(caminho_modelo)

clf = dados_modelo["modelo"]
colunas_esperadas = dados_modelo["colunas"]
num_features = dados_modelo["num_features"]

print(f"Modelo carregado! Esperando {num_features} features de entrada (1 ou 2 mãos).")

# Fila para estabilizar a predição na tela (guarda os últimos 7 frames)
historico_sinais = deque(maxlen=7)

# ==============================================================================
# 2. CONFIGURAÇÃO DO MEDIAPIPE HANDS (SUPORTE ATÉ 2 MÃOS) E WEBCAM
# ==============================================================================
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,  # Permite reconhecer sinais de 1 mão (letras) ou 2 mãos (ex: CASA)
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if not cap.isOpened():
    raise RuntimeError("Erro ao acessar a webcam.")

print("Reconhecimento iniciado! Pressione 'q' na janela de vídeo para sair.")

# ==============================================================================
# 3. LOOP DE CAPTURA E INFERÊNCIA EM TEMPO REAL
# ==============================================================================
while cap.isOpened():
    sucesso, frame = cap.read()
    if not sucesso:
        break

    # Espelha o frame horizontalmente (idêntico ao coletor)
    frame = cv2.flip(frame, 1)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    resultado = hands.process(frame_rgb)

    texto_exibicao = "Aguardando sinal..."
    cor_box = (0, 0, 255)  # Vermelho

    if resultado.multi_hand_landmarks:
        # Ordena as mãos da esquerda para a direita para manter a mesma ordem do treino
        maos_ordenadas = sorted(resultado.multi_hand_landmarks, key=lambda h: h.landmark[0].x)
        coordenadas = []

        for hand_landmarks in maos_ordenadas[:2]:
            # Desenha as conexões das mãos na tela
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

            # Normalização pelo pulso (Ponto 0) de cada mão
            pulso_x = hand_landmarks.landmark[0].x
            pulso_y = hand_landmarks.landmark[0].y
            pulso_z = hand_landmarks.landmark[0].z

            for lm in hand_landmarks.landmark:
                coordenadas.extend([lm.x - pulso_x, lm.y - pulso_y, lm.z - pulso_z])

        # Se apenas 1 mão apareceu na tela e o modelo usa 126 colunas, preenche a 2ª mão com zeros
        if len(maos_ordenadas) == 1 and num_features == 126:
            coordenadas.extend([0.0] * 63)

        # Ajusta o tamanho exato caso o modelo seja de 63 ou 126 features
        coordenadas = coordenadas[:num_features]

        # Monta o DataFrame com os nomes exatos das colunas do treinamento
        X_input = pd.DataFrame([coordenadas], columns=colunas_esperadas)

        # Faz a predição e calcula a probabilidade no Random Forest
        predicao = clf.predict(X_input)[0]
        probabilidades = clf.predict_proba(X_input)[0]
        confianca = np.max(probabilidades)

        if confianca >= 0.45:
            historico_sinais.append(predicao)
            # Pega o sinal mais frequente nos últimos frames para evitar oscilação
            sinal_estavel = Counter(historico_sinais).most_common(1)[0][0]
            texto_exibicao = f"Sinal: {sinal_estavel} ({confianca*100:.0f}%)"
            cor_box = (0, 200, 0)  # Verde
        else:
            texto_exibicao = f"Incerto ({confianca*100:.0f}%)"
            cor_box = (0, 165, 255)  # Laranja
    else:
        historico_sinais.clear()

    # Desenha o painel superior para exibir o sinal ou palavra reconhecida
    cv2.rectangle(frame, (10, 10), (480, 65), (30, 30, 30), -1)
    cv2.putText(frame, texto_exibicao, (20, 48),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, cor_box, 2, cv2.LINE_AA)

    cv2.imshow("Reconhecimento de Letras e Palavras - Libras", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
hands.close()
print("Sessão encerrada.")

Carregando modelo de: modelo_asl_rf.pkl...
Modelo carregado! Esperando 126 features de entrada (1 ou 2 mãos).
Reconhecimento iniciado! Pressione 'q' na janela de vídeo para sair.
Sessão encerrada.
